# Unifier Data Warehouse & API Simplify Data Access With A Single Interface

In [1]:
import os
import pandas as pd
from IPython.display import display

### If you have not installed unifier you can do so by running the following command:

In [2]:
#pip install unifier

## Simply Import and initialize Unifier and Go!

In [3]:
from unifier import unifier as Unifier

In [4]:
Unifier.user = 'demo'
Unifier.token ='qZy7KpE+L5Fg2MmjFqC/1xmweo7pgnnACCcK3JNEAec='
os.environ['UNIFIER_USER'] = Unifier.user
os.environ['UNIFIER_TOKEN'] = Unifier.token

# Xtech Security Master & Corporate Actions Documentation

## Overview
**Xtech Security Master & Corporate Actions** connects enduring security identity with the events that change a security over time. It combines security and listing states, time-aware symbol and identifier mappings, corporate action records, adjustment factors, and explanatory event notes. Effective dates describe when a fact applied; knowledge dates preserve when that fact or revision became available. Analysts can trace an event from its security to the published factor and related note, then reconstruct the information available at a chosen historical date. This connected history supports dependable security joins, event research, price and share-count adjustment workflows, and reproducible backtests.

### Key Details
- **Product**: Xtech Security Master & Corporate Actions
- **Coverage**: Securities and listings in US markets
- **Delivery Frequency**: Daily updates
- **Delivery Time**: T+1 9:30 AM UTC
- **Delivery Method**: Unifier API
- **Data Frequency**: Daily
- **Data Description**: Security Master, Symbology, Corporate Actions, Adjustment Factors, and Event Notes
- **Join Keys**: `SecID` connects security-level records; `EventID` connects actions, factors, and notes
- **Deep History**: Valid history from 2005-01-01; knowledge history from 2016-01-01

---

## Datasets

| **Dataset** | **Unifier name** | **`key`** | **`back_to` / `up_to` filter on** |
|---|---|---|---|
| Security Master | `data_hub_v2_0_security_master` | `SecID` | `valid_from` |
| Symbology | `data_hub_v2_0_symbology` | `SecID` | `valid_from` |
| Corporate Actions | `data_hub_v2_0_corp_actions` | `SecID` | `EvtChangeDT` |
| Adjustment Factors | `data_hub_v2_0_adjustment_factors` | `SecID` | `ExDate` |
| Event Notes | `data_hub_v2_0_event_notes` | `EventID` | `NtsChangeDT` |

### How to read the data
- Each daily delivery is a complete file that carries every recorded version on both time axes. Without `asof_date` you read the latest delivery; with `asof_date` you read the delivery as it stood on that date.
- The point-in-time view for any historical date is a filter on the columns: `valid_from` / `valid_to` for when a fact applied, `knowledge_from` / `knowledge_to` for when it was known. Intervals are half-open, and `2262-01-01` marks an open end.
- Column names are returned in lower case (`secid`, `issuername`, ...) and dates are returned as strings.

---

### Why This Product is Unique
- **Stable Identity**: A stable `SecID` keeps security identity separate from changing market symbols.
- **Two Time Axes**: Effective and knowledge dates distinguish when a fact was true from when it was known.
- **Linked Records**: Corporate actions, factor revisions, and event notes are linked through security and event identifiers.
- **Reproducible Backtests**: Historical security mapping and action analysis can be reproduced, and revised information is kept out of a backtest until it was available.

---

### Potential Use Cases
- Resolve a historical ticker, CUSIP, or ISIN to the correct security for a chosen date.
- Track changes in issuer name, listing status, and exchange over time.
- Study splits, dividends, and other corporate actions alongside the affected security.
- Select the factor revision that was known at the time of a historical backtest.
- Investigate factor changes with linked event notes.
- Build event studies with consistent security and event identifiers.
- Audit historical joins and reduce errors from ticker changes or identifier reuse.

---

### Field Descriptions

#### Security Master (30 columns)
Listing and security state on effective and knowledge time axes.

| **Column Name** | **Data Type** | **Description** |
|---|---|---|
| `SecID` | string | Stable security identifier shared by listings and related records. |
| `IssID` | int64 | Issuer identifier used to group securities of the same issuer. |
| `IssuerName` | str | Issuer name carried with this record. |
| `SecurityDescription` | str | Vendor description of the security. |
| `Isin` | str | International Securities Identification Number. |
| `USCode` | str | US CUSIP identifier. |
| `LocalCode` | str | Trading or listing code used in the local market. |
| `ExchgCD` | str | Vendor exchange code for the listing. |
| `PrimaryExchgCD` | str | Vendor code for the primary listing exchange. |
| `ListCntryCD` | str | Country code of the listing market. |
| `ListStatusCD` | str | Code describing the listing activity status. |
| `entity_key` | int64 | Listing-level key used to group all states of one listing. |
| `valid_from` | datetime[ns] | Start of the interval when the mapping or state was effective. |
| `valid_to` | datetime[ns] | End of that effective interval; an open end means still valid. |
| `knowledge_from` | datetime[ns,UTC] | UTC time when this version became available. |
| `knowledge_to` | datetime[ns,UTC] | UTC time when this version was superseded or closed. |
| `closed_by` | str | Why the knowledge version closed, such as update or delete. |
| `valid_closed_by` | str | Why the effective interval ended, such as update or delete. |
| `source` | str | Source feed used to construct the record. |
| `source_file` | str | Source file or partition from which the record was built. |
| `knowledge_left_censored` | bool | True when earlier changes may predate the captured history. |
| `Sedol` | str | SEDOL security identifier. |
| `Figi` | str | Bloomberg Financial Instrument Global Identifier. |
| `FigiTicker` | str | Ticker associated with the FIGI listing. |
| `BbgCompID` | str | Bloomberg composite global identifier. |
| `BbgCompTicker` | str | Bloomberg composite ticker. |
| `OperationalMic` | str | Market Identifier Code for the operating exchange. |
| `SectyCD` | str | Vendor code for security type. |
| `CIK` | str | SEC Central Index Key for the issuer. |
| `LEI` | str | Legal Entity Identifier for the issuer. |

#### Symbology (8 columns)
Identifier mappings over effective and knowledge time.

| **Column Name** | **Data Type** | **Description** |
|---|---|---|
| `id_type` | str | Identifier namespace, such as ticker, ISIN, CUSIP, or FIGI. |
| `id_value` | string | Identifier value within that namespace. |
| `SecID` | string | Security to which this identifier mapped during its effective interval. |
| `entity_key` | int64 | Listing-level key inherited from the security master. |
| `valid_from` | datetime[ns] | Start of the interval when the mapping or state was effective. |
| `valid_to` | datetime[ns] | End of that effective interval; an open end means still valid. |
| `knowledge_time` | datetime[ns,UTC] | UTC time when the identifier mapping became known. |
| `source` | str | Source feed used to construct the record. |

#### Corporate Actions (37 columns)
Versioned corporate action events.

| **Column Name** | **Data Type** | **Description** |
|---|---|---|
| `EventCD` | str | Vendor event code; pairs with EventID to identify an event. |
| `EventID` | string | Vendor event identifier; use with EventCD for action and note joins. |
| `EvtUniqueID` | int64 | Vendor event-row ID; unique with the listing record ID. |
| `EvtActionCD` | str | Vendor action code for an event insert, update, or closure. |
| `EventSubTypeCD` | str | Vendor code for a more specific event category. |
| `RelatedEventCD` | str | Event code of a linked corporate action. |
| `RelatedEventID` | float64 | Event ID of a linked action, used with RelatedEventCD. |
| `EventCreateDT` | object | Date the event was first created by the vendor. |
| `EvtChangeDT` | datetime[ns] | Vendor timestamp of the event-row change. |
| `ExDT` | object | Ex-date associated with the event. |
| `RecordDT` | object | Record date for determining entitled holders. |
| `PayDT` | object | Scheduled payment date. |
| `EffectiveDT` | object | Date the action takes effect. |
| `RatioOld` | float64 | Old-holding amount or denominator of an action ratio. |
| `RatioNew` | float64 | New-share amount or numerator of an action ratio. |
| `OptionID` | float64 | Option number for an event with alternative payment choices. |
| `SerialID` | float64 | Serial number for a component within an event option. |
| `SecID` | string | Security affected by this event row. |
| `IssID` | int64 | Issuer identifier used to group securities of the same issuer. |
| `IssuerName` | str | Issuer name carried with this record. |
| `Isin` | str | International Securities Identification Number. |
| `USCode` | str | US CUSIP identifier. |
| `SectyCD` | str | Vendor code for security type. |
| `LstUniqueID` | int64 | Vendor listing-record ID; part of the event-row identity. |
| `ScexhID` | int64 | Vendor listing-level identifier. |
| `ExchgCD` | str | Vendor exchange code for the listing. |
| `ListCntryCD` | str | Country code of the listing market. |
| `LocalCode` | str | Trading or listing code used in the local market. |
| `OutSecID` | float64 | Vendor security ID for an outturn security. |
| `OutIsin` | str | ISIN of an outturn security. |
| `OutUSCode` | str | CUSIP of an outturn security. |
| `OutFigi` | str | FIGI of an outturn security. |
| `entity_key` | str | Composite key for one event row, listing, and payment choice. |
| `knowledge_from` | datetime[ns,UTC] | UTC time when this version became available. |
| `knowledge_to` | datetime[ns,UTC] | UTC time when this version was superseded or closed. |
| `closed_by` | str | Why the knowledge version closed, such as update or delete. |
| `knowledge_left_censored` | bool | True when earlier changes may predate the captured history. |

#### Adjustment Factors (15 columns)
Published adjustment factors and revisions.

| **Column Name** | **Data Type** | **Description** |
|---|---|---|
| `SecID` | string | Stable security identifier shared by listings and related records. |
| `EventID` | string | Action identifier paired with SecID in this factor log. |
| `EventType` | str | Action type carried by the adjustment feed. |
| `Reason` | str | Vendor reason code used to classify the adjustment. |
| `IssuerName` | str | Issuer name carried with this record. |
| `ExDate` | datetime[ns] | Ex-date used to determine when the factor applies. |
| `Factor` | float64 | Published multiplicative adjustment factor for this event version. |
| `asof_date` | object / str | Source publication or processing date of the record. |
| `first_seen_date` | object / str | First date this event appeared in the adjustment feed. |
| `entity_key` | str | Composite SecID and EventID key for factor revisions. |
| `knowledge_from` | datetime[ns,UTC] | UTC time when this version became available. |
| `knowledge_to` | datetime[ns,UTC] | UTC time when this version was superseded or closed. |
| `closed_by` | str | Whether this factor version ended by update or retraction. |
| `revision` | int64 | Zero-based sequence number of published factor changes. |
| `source_file` | str | Source file or partition from which the record was built. |

#### Event Notes (13 columns)
Versioned vendor narratives attached to events.

| **Column Name** | **Data Type** | **Description** |
|---|---|---|
| `EventCD` | str | Code paired with EventID to identify an event. |
| `EventID` | string | Event ID; pair with EventCD to link to the action. |
| `ReflinkID` | int64 | Reference ID distinguishing several notes for the same event. |
| `NotesType` | str | Type of event narrative. |
| `NtsActionCD` | str | Vendor action code for an event note. |
| `NtsChangeDT` | datetime[ns] | Vendor timestamp of the note change. |
| `NotesText` | str | Full vendor narrative for the event note. |
| `entity_key` | str | Composite key for one note and its revisions. |
| `knowledge_from` | datetime[ns,UTC] | UTC time when this version became available. |
| `knowledge_to` | datetime[ns,UTC] | UTC time this note version closed. |
| `closed_by` | str | Update or delete that closed this version. |
| `knowledge_left_censored` | bool | Earlier note versions may be missing. |
| `source_file` | str | Original note file or partition. |

---


## Security Master: retrieve the full history of a security using key (SecID)

Meta Platforms (SecID `3513095`) listed as Facebook Inc under the ticker `FB`, was renamed Meta Platforms on 2021-10-28, and switched its ticker to `META` on 2022-06-09. The SecID never changed. Rows that share a `valid_from` / `valid_to` interval are successive knowledge versions of the same state; here the later versions added the FIGI and the CIK.

In [5]:
sm = Unifier.get_dataframe(name='data_hub_v2_0_security_master', key='3513095')
display(sm.sort_values(['valid_from', 'knowledge_from'])[
    ['secid', 'issuername', 'localcode', 'isin', 'exchgcd', 'liststatuscd', 'figi', 'cik',
     'valid_from', 'valid_to', 'knowledge_from', 'knowledge_to', 'closed_by']
])

,secid,issuername,localcode,isin,exchgcd,liststatuscd,figi,cik,valid_from,valid_to,knowledge_from,knowledge_to,closed_by
0,3513095,Facebook Inc,FB,US30303M1027,USNASD,L,NaN,NaN,2012-05-18 00:00:00.000,2021-10-28 00:00:00.000,2019-02-20 15:00:00.000,2262-01-01 00:00:00.000,NaN
1,3513095,Meta Platforms Inc,FB,US30303M1027,USNASD,L,NaN,NaN,2021-10-28 00:00:00.000,2022-06-09 00:00:00.000,2022-06-09 03:41:35.000,2262-01-01 00:00:00.000,NaN
2,3513095,Meta Platforms Inc,META,US30303M1027,USNASD,L,NaN,NaN,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,2022-06-01 13:19:12.000,2024-04-01 17:27:48.000,update
3,3513095,Meta Platforms Inc,META,US30303M1027,USNASD,L,BBG002GPKKZ7,NaN,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,2024-04-01 17:27:48.000,2024-08-02 16:32:13.000,update
4,3513095,Meta Platforms Inc,META,US30303M1027,USNASD,L,BBG002GPKKZ7,0001326801,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,2024-08-02 16:32:13.000,2025-12-30 17:26:39.000,update
5,3513095,Meta Platforms Inc,META,US30303M1027,USNASD,L,BBG002GPKKZ7,0001326801,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,2025-12-30 17:26:39.000,2262-01-01 00:00:00.000,NaN


## Security Master: retrieve the records that took effect within a date range using back_to and up_to

For the security master, `back_to` / `up_to` filter on `valid_from`. This returns the states of Meta that began in 2021–2022: the rename and the ticker change.

In [6]:
df = Unifier.get_dataframe(name='data_hub_v2_0_security_master', key='3513095', back_to='2021-01-01', up_to='2022-12-31')
display(df.sort_values(['valid_from', 'knowledge_from'])[
    ['secid', 'issuername', 'localcode', 'valid_from', 'valid_to', 'knowledge_from', 'knowledge_to', 'closed_by']
])

,secid,issuername,localcode,valid_from,valid_to,knowledge_from,knowledge_to,closed_by
0,3513095,Meta Platforms Inc,FB,2021-10-28 00:00:00.000,2022-06-09 00:00:00.000,2022-06-09 03:41:35.000,2262-01-01 00:00:00.000,NaN
1,3513095,Meta Platforms Inc,META,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,2022-06-01 13:19:12.000,2024-04-01 17:27:48.000,update
2,3513095,Meta Platforms Inc,META,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,2024-04-01 17:27:48.000,2024-08-02 16:32:13.000,update
3,3513095,Meta Platforms Inc,META,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,2024-08-02 16:32:13.000,2025-12-30 17:26:39.000,update
4,3513095,Meta Platforms Inc,META,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,2025-12-30 17:26:39.000,2262-01-01 00:00:00.000,NaN


## Security Master: reconstruct what was known on a historical date

Filter on both time axes: the state effective on `valid_date`, as it was known on `knowledge_date`. On 2023-01-03 the Meta record carried no FIGI or CIK; both arrived in 2024. A backtest dated 2023-01-03 should see the first view below, not the second.

In [7]:
for col in ['valid_from', 'valid_to', 'knowledge_from', 'knowledge_to']:
    sm[col] = pd.to_datetime(sm[col])

def as_of(df, valid_date, knowledge_date):
    valid_date, knowledge_date = pd.Timestamp(valid_date), pd.Timestamp(knowledge_date)
    return df[(df['valid_from'] <= valid_date) & (df['valid_to'] > valid_date)
              & (df['knowledge_from'] <= knowledge_date) & (df['knowledge_to'] > knowledge_date)]

columns = ['secid', 'issuername', 'localcode', 'figi', 'cik', 'knowledge_from', 'knowledge_to']
display(as_of(sm, valid_date='2023-01-03', knowledge_date='2023-01-03')[columns])        # as known at the time
display(as_of(sm, valid_date='2023-01-03', knowledge_date=pd.Timestamp.now())[columns])  # as known today

,secid,issuername,localcode,figi,cik,knowledge_from,knowledge_to
2,3513095,Meta Platforms Inc,META,NaN,NaN,2022-06-01 13:19:12,2024-04-01 17:27:48


,secid,issuername,localcode,figi,cik,knowledge_from,knowledge_to
5,3513095,Meta Platforms Inc,META,BBG002GPKKZ7,0001326801,2025-12-30 17:26:39,2262-01-01


## Symbology: resolve a ticker to the right security at a point in time

The ticker `META` has belonged to three different securities. `column_filters` matches on any column, and the `valid_from` / `valid_to` interval then picks the security that held the ticker on a given date. On 2015-01-02 nobody held it.

In [8]:
meta = Unifier.get_dataframe(name='data_hub_v2_0_symbology', column_filters="id_value = 'META'")
display(meta)

,id_type,id_value,secid,entity_key,valid_from,valid_to,knowledge_time,source,asof_date
0,ticker,META,39742,159235,2005-01-01 00:00:00.000,2005-07-27 00:00:00.000,2026-09-29 00:00:00.000,srf,2026-09-28
1,ticker,META,6981246,9214008,2021-06-30 00:00:00.000,2022-01-31 00:00:00.000,2026-09-29 00:00:00.000,srf,2026-09-28
2,ticker,META,3513095,5373265,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,2026-09-29 00:00:00.000,ref_lst,2026-09-28


In [9]:
meta['valid_from'] = pd.to_datetime(meta['valid_from'])
meta['valid_to'] = pd.to_datetime(meta['valid_to'])

for date in ['2005-03-01', '2015-01-02', '2022-01-03', '2023-01-03']:
    d = pd.Timestamp(date)
    print(date, '->', meta[(meta['valid_from'] <= d) & (meta['valid_to'] > d)]['secid'].tolist())

2005-03-01 -> ['39742']
2015-01-02 -> []
2022-01-03 -> ['6981246']
2023-01-03 -> ['3513095']


## Security Master: look up several securities at once using keys

Pass the SecIDs found above as `keys` to see who they are: Metatec International on the OTC market in 2005, an ETF issued by Listed Funds Trust in 2021 (it moved to `METV` on 2022-01-31), and Meta Platforms from 2022-06-09.

In [10]:
df = Unifier.get_dataframe(name='data_hub_v2_0_security_master', keys=meta['secid'].unique().tolist())
display(df[df['localcode'] == 'META'][
    ['secid', 'issuername', 'localcode', 'exchgcd', 'valid_from', 'valid_to']
].drop_duplicates().sort_values('valid_from'))

,secid,issuername,localcode,exchgcd,valid_from,valid_to
0,39742,"Metatec International, Inc.",META,USOTC,2005-01-01 00:00:00.000,2005-07-27 00:00:00.000
10,6981246,Listed Funds Trust,META,USPAC,2021-06-30 00:00:00.000,2022-01-31 00:00:00.000
9,3513095,Meta Platforms Inc,META,USNASD,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000


## Symbology: list every identifier a security has carried using key (SecID)

One row per identifier and effective interval: ticker, ISIN, CUSIP, FIGI, CIK, LEI and more, each with the dates it applied.

In [11]:
df = Unifier.get_dataframe(name='data_hub_v2_0_symbology', key='3513095')
display(df.sort_values(['id_type', 'valid_from'])[['id_type', 'id_value', 'secid', 'valid_from', 'valid_to', 'source']])

,id_type,id_value,secid,valid_from,valid_to,source
9,bbg_composite_id,BBG000MM2P62,3513095,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,ref_lst
10,bbg_composite_ticker,META US,3513095,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,ref_lst
14,cik,0001326801,3513095,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,ref_lst
11,cusip,30303M102,3513095,2012-05-18 00:00:00.000,2021-10-28 00:00:00.000,srf
12,cusip,30303M102,3513095,2021-10-28 00:00:00.000,2022-06-09 00:00:00.000,srf
13,cusip,30303M102,3513095,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,ref_lst
7,figi,BBG002GPKKZ7,3513095,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,ref_lst
8,figi_ticker,META UW,3513095,2022-06-09 00:00:00.000,2262-01-01 00:00:00.000,ref_lst
1,isin,US30303M1027,3513095,2012-05-18 00:00:00.000,2021-10-28 00:00:00.000,srf
2,isin,US30303M1027,3513095,2021-10-28 00:00:00.000,2022-06-09 00:00:00.000,srf


## Corporate Actions: retrieve a security's events within a date range using key and back_to / up_to

For corporate actions, `back_to` / `up_to` filter on the vendor change timestamp `EvtChangeDT`. Visa (SecID `787527`) in Q4 2022 had a quarterly dividend (EventID `2732899`) and an AGM. The sections below follow the dividend through the factor log and the event notes.

In [12]:
visa_actions = Unifier.get_dataframe(name='data_hub_v2_0_corp_actions', key='787527', back_to='2022-10-01', up_to='2022-12-31')
display(visa_actions[
    ['secid', 'issuername', 'localcode', 'eventcd', 'eventid', 'evtactioncd', 'evtchangedt',
     'exdt', 'recorddt', 'paydt', 'effectivedt', 'knowledge_from', 'knowledge_to']
])

,secid,issuername,localcode,eventcd,eventid,evtactioncd,evtchangedt,exdt,recorddt,paydt,effectivedt,knowledge_from,knowledge_to
0,787527,Visa Inc,V,DIV,2732899,U,2022-11-08 13:10:38.000,2022-11-09,2022-11-11,2022-12-01,NaN,2022-11-08 17:29:44.000,2262-01-01 00:00:00.000
1,787527,Visa Inc,V,AGM,1121748,I,2022-12-02 17:27:27.000,NaN,2022-11-25,NaN,2023-01-24,2022-12-02 17:27:27.000,2262-01-01 00:00:00.000


## Adjustment Factors: retrieve factors within an ex-date range using key and back_to / up_to

For adjustment factors, `back_to` / `up_to` filter on `ExDate`. Apple's (SecID `33449`) 4-for-1 split on 2020-08-31 has `Factor` 0.25: multiply prices before the ex-date by it. A split also restates the share count, so historical volume is divided by it; the cash dividend (`Reason` 17) moves price only.

The split also shows a revision. On 2026-09-02 the vendor reclassified it (`EventType` SD → BON, `Reason` 5 → 13) with the same factor, so the original version was closed (`closed_by` = update) and revision 1 opened.

In [13]:
df = Unifier.get_dataframe(name='data_hub_v2_0_adjustment_factors', key='33449', back_to='2020-08-01', up_to='2020-09-30')
display(df.sort_values(['exdate', 'revision'])[
    ['secid', 'issuername', 'eventid', 'eventtype', 'reason', 'exdate', 'factor',
     'revision', 'knowledge_from', 'knowledge_to', 'closed_by']
])

,secid,issuername,eventid,eventtype,reason,exdate,factor,revision,knowledge_from,knowledge_to,closed_by
1,33449,Apple Inc,1978520,DIV,17,2020-08-07 00:00:00.000,0.9982,0,2020-08-07 00:00:00.000,2262-01-01 00:00:00.000,NaN
0,33449,Apple Inc,6705964,SD,5,2020-08-31 00:00:00.000,0.2500,0,2020-08-31 00:00:00.000,2026-09-02 00:00:00.000,update
2,33449,Apple Inc,6705964,BON,13,2020-08-31 00:00:00.000,0.2500,1,2026-09-02 00:00:00.000,2262-01-01 00:00:00.000,NaN


## Adjustment Factors: retrieve the file as delivered on a specific date using asof_date

`asof_date` reads an earlier daily delivery. The 2026-09-01 delivery predates the reclassification: the split is still SD at revision 0, and its knowledge interval is still open.

In [14]:
df = Unifier.get_dataframe(name='data_hub_v2_0_adjustment_factors', key='33449', asof_date='2026-09-01', back_to='2020-08-01', up_to='2020-09-30')
display(df.sort_values(['exdate', 'revision'])[
    ['secid', 'issuername', 'eventid', 'eventtype', 'reason', 'exdate', 'factor',
     'revision', 'knowledge_from', 'knowledge_to', 'closed_by', 'asof_date']
])

,secid,issuername,eventid,eventtype,reason,exdate,factor,revision,knowledge_from,knowledge_to,closed_by,asof_date
0,33449,Apple Inc,1978520,DIV,17,2020-08-07 00:00:00.000,0.9982,0,2020-08-07 00:00:00.000,2262-01-01 00:00:00.000,None,2026-09-01
1,33449,Apple Inc,6705964,SD,5,2020-08-31 00:00:00.000,0.2500,0,2020-08-31 00:00:00.000,2262-01-01 00:00:00.000,None,2026-09-01


## Adjustment Factors: select the factor revision known at a historical date

Visa's dividend (EventID `2732899`) was published three times: ex-date 2022-11-09, revised to 2022-11-10 on 2022-11-11, and restored on 2022-11-14. A backtest dated 2022-11-12 must use revision 1, the value that was live then, not the value known today.

In [15]:
visa_factors = Unifier.get_dataframe(name='data_hub_v2_0_adjustment_factors', key='787527', back_to='2022-11-01', up_to='2022-11-30')
display(visa_factors.sort_values('revision')[
    ['secid', 'eventid', 'eventtype', 'exdate', 'factor', 'revision', 'knowledge_from', 'knowledge_to', 'closed_by']
])

,secid,eventid,eventtype,exdate,factor,revision,knowledge_from,knowledge_to,closed_by
0,787527,2732899,DIV,2022-11-09 00:00:00.000,0.99777,0,2022-11-09 00:00:00.000,2022-11-11 00:00:00.000,update
1,787527,2732899,DIV,2022-11-10 00:00:00.000,0.99768,1,2022-11-11 00:00:00.000,2022-11-14 00:00:00.000,update
2,787527,2732899,DIV,2022-11-09 00:00:00.000,0.99777,2,2022-11-14 00:00:00.000,2262-01-01 00:00:00.000,NaN


In [16]:
knowledge_date = pd.Timestamp('2022-11-12')
known = visa_factors[(pd.to_datetime(visa_factors['knowledge_from']) <= knowledge_date)
                     & (pd.to_datetime(visa_factors['knowledge_to']) > knowledge_date)]
display(known[['eventid', 'exdate', 'factor', 'revision', 'knowledge_from', 'knowledge_to']])

,eventid,exdate,factor,revision,knowledge_from,knowledge_to
1,2732899,2022-11-10 00:00:00.000,0.99768,1,2022-11-11 00:00:00.000,2022-11-14 00:00:00.000


## Event Notes: read the vendor narrative for an event using key (EventID)

Event notes are keyed by `EventID` and carry no `SecID`: a note belongs to the event. The vendor writes `<BR>` line breaks inside `NotesText`.

In [17]:
visa_notes = Unifier.get_dataframe(name='data_hub_v2_0_event_notes', key='2732899')
display(visa_notes[['eventcd', 'eventid', 'notestype', 'ntschangedt', 'knowledge_from', 'knowledge_to']])
print(visa_notes['notestext'].iloc[0].replace('<BR>', '\n'))

,eventcd,eventid,notestype,ntschangedt,knowledge_from,knowledge_to
0,DIV,2732899,Notes,2022-11-08 13:10:38.000,2022-11-08 17:29:44.000,2262-01-01 00:00:00.000


(As on 07/11/2022) ATCA
Issuer : VISA INC-CLASS A SHARES
Isin : US92826C8394
Ex date : 11/09/2022


## Trace one event across the product using SecID and EventID

Corporate actions join adjustment factors on `SecID` + `EventID`, and event notes on `EventCD` + `EventID` (`EventID` alone can collide across event types). One row per factor revision of the Visa dividend, next to the action's dates and its note.

In [18]:
trace = (
    visa_actions[['secid', 'eventcd', 'eventid', 'exdt', 'recorddt', 'paydt']]
    .merge(visa_factors[['secid', 'eventid', 'revision', 'exdate', 'factor', 'knowledge_from']], on=['secid', 'eventid'])
    .merge(visa_notes[['eventcd', 'eventid', 'notestype', 'ntschangedt']], on=['eventcd', 'eventid'])
)
display(trace.sort_values('revision'))

,secid,eventcd,eventid,exdt,recorddt,paydt,revision,exdate,factor,knowledge_from,notestype,ntschangedt
0,787527,DIV,2732899,2022-11-09,2022-11-11,2022-12-01,0,2022-11-09 00:00:00.000,0.99777,2022-11-09 00:00:00.000,Notes,2022-11-08 13:10:38.000
1,787527,DIV,2732899,2022-11-09,2022-11-11,2022-12-01,1,2022-11-10 00:00:00.000,0.99768,2022-11-11 00:00:00.000,Notes,2022-11-08 13:10:38.000
2,787527,DIV,2732899,2022-11-09,2022-11-11,2022-12-01,2,2022-11-09 00:00:00.000,0.99777,2022-11-14 00:00:00.000,Notes,2022-11-08 13:10:38.000
